# 1. Imports

In [1]:
# ============================================
# 1. Librerías
# ============================================
import torch, pandas as pd, numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import polars as pl
import sys

sys.path.append('../src')  
from models.nn_multihead_model import BikeDestETAEnhanced              # <-- tu archivo con la clase

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
torch.manual_seed(SEED);  np.random.seed(SEED)

# 2. DF

In [2]:
df = pd.read_parquet("../data/processed/ecobici_features.parquet")

In [3]:
df.columns

Index(['id_recorrido', 'duracion_recorrido', 'fecha_origen_recorrido',
       'id_estacion_origen', 'nombre_estacion_origen',
       'direccion_estacion_origen', 'long_estacion_origen',
       'lat_estacion_origen', 'fecha_destino_recorrido', 'id_estacion_destino',
       ...
       'peak_commute', 'dep_ma_24h', 'dep_std_24h', 'dep_ratio_DT_24h',
       'near_dep_sum_DT', 'near_dep_lag_1', 'hour', 'dow', 'month', 'day'],
      dtype='object', length=130)

In [4]:
# --------------------------------------------
# LISTAS DEFINITIVAS DE FEATURES
# --------------------------------------------

cat_cols = [
    # ───────── variables categóricas originales ─────────
    "id_estacion_origen",
    "modelo_bicicleta",
    "genero",
    "weather_weather_code_x",
    "weather_is_day_x",
    # ───────── flags e indicadores ─────────
    "is_holiday_ar",
    "is_weekend",
    "payday_flag",
    "vacation_season",
    "peak_commute",
    "precip_flag",
    # ───────── calendarios (enteros) ─────────
    "hour",
    "dow",
    "month",
    "day"
]

cont_cols = [
    # ───────── ubicación ─────────
    "long_estacion_origen", "lat_estacion_origen",
    # ───────── tiempo (t-1) ─────────
    # "weather_temperature_2m_x", "weather_relative_humidity_2m_x",
    # "weather_dew_point_2m_x", "weather_apparent_temperature_x",
    # "weather_precipitation_x", "weather_rain_x",
    # "weather_pressure_msl_x", "weather_surface_pressure_x",
    # "weather_cloud_cover_x", "weather_cloud_cover_low_x",
    # "weather_cloud_cover_mid_x", "weather_cloud_cover_high_x",
    # "weather_et0_fao_evapotranspiration_x",
    # "weather_vapour_pressure_deficit_x",
    # "weather_wind_speed_10m_x",
    # "weather_wind_gusts_10m_x",
    # "weather_soil_temperature_0_to_7cm_x",
    # "weather_soil_moisture_0_to_7cm_x",
    # "weather_sunshine_duration_x", "weather_direct_radiation_x",
    # # ───────── variables ingenierizadas del viento ─────────
    # "wind_dir_sin", "wind_dir_cos",
    # # ───────── lags/rolling de precipitación ─────────
    # "precipitation_lag_1h", "rain_lag_1h",
    # # ───────── one-hot del código de clima ─────────
    # "weather_code_cat_Clear", "weather_code_cat_Clouds",
    # "weather_code_cat_Drizzle", "weather_code_cat_Rain",
    # # ───────── demanda y estadísticos de operación ─────────
    # "dep_last_DT", "trip_dur_mean_last_DT",
    # "model_FIT_cnt", "model_ICONIC_cnt",
    # "share_male", "share_female", "share_other",
    "dep_lag_1", "dep_lag_2", "dep_lag_3",
    "dep_lag_4", "dep_lag_5", "dep_lag_6",
    "arr_last_DT", "arr_lag_1", "arr_lag_2",
    "arr_lag_3", "arr_lag_4", "arr_lag_5", "arr_lag_6",
    # ───────── rolling 24 h y proximidad ─────────
    "dep_ma_24h", "dep_std_24h", "dep_ratio_DT_24h",
    "near_dep_sum_DT", "near_dep_lag_1",
    # ───────── codificaciones cíclicas ─────────
    "sin_hour", "cos_hour",
    "sin_dow",  "cos_dow",
    "sin_month","cos_month"
]

In [5]:
target_cls = 'id_estacion_origen'
target_eta = 'duracion_recorrido'

In [6]:
# Focus on key columns for our prediction task
print("=== TRIPS DATA ANALYSIS ===")
print(f"Trips dataset has {df.shape[0]} rows and {df.shape[1]} columns")

# Check unique stations
if 'id_estacion_origen' in df.columns and 'id_estacion_destino' in df.columns:
    print(f"Unique origin stations: {df['id_estacion_origen'].unique().shape[0]}")
    print(f"Unique destination stations: {df['id_estacion_destino'].unique().shape[0]}")

# Check time columns
time_cols = [col for col in df.columns if 'fecha' in col.lower() or 'tiempo' in col.lower() or 'hora' in col.lower()]
print(f"Time-related columns: {time_cols}")

# Check if we have trip duration information
duration_cols = [col for col in df.columns if 'duracion' in col.lower() or 'duration' in col.lower()]
print(f"Duration columns: {duration_cols}")

=== TRIPS DATA ANALYSIS ===
Trips dataset has 12976053 rows and 130 columns
Unique origin stations: 501
Unique destination stations: 502
Time-related columns: ['fecha_origen_recorrido', 'fecha_destino_recorrido']
Duration columns: ['duracion_recorrido', 'weather_sunshine_duration_x', 'weather_sunshine_duration_y']


In [7]:
# Calculate departures and arrivals per station
departures = df['id_estacion_origen'].value_counts()
arrivals = df['id_estacion_destino'].value_counts()

# Find stations with at least 50 departures and 50 arrivals
valid_stations = set(departures[departures >= 50].index) & set(arrivals[arrivals >= 50].index)

print(f'Number of total stations: {len(departures)}')
print(f'Number of valid stations: {len(valid_stations)}')

# Filter dataframe to keep only trips where both origin and destination are valid stations
df = df[
    df['id_estacion_origen'].isin(valid_stations) &
    df['id_estacion_destino'].isin(valid_stations)
]

print(f"Filtered dataframe shape: {df.shape}")
print(f"Number of valid stations: {len(valid_stations)}")

Number of total stations: 501
Number of valid stations: 495
Filtered dataframe shape: (12975362, 130)
Number of valid stations: 495


## 2.2 Encoding

In [8]:
# ---------- 2.2 Encoding de categóricas ----------
encoders = {}
for col in cat_cols + [target_cls]:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

## 2.3 val y test (cambiar por la fecha)

In [9]:
import gc

In [10]:
gc.collect()

0

In [11]:
# ---------- 2.3 Train / Val / Test ----------
train_df, test_df = train_test_split(df, test_size=0.15, shuffle=False)   # corte temporal
train_df, val_df  = train_test_split(train_df, test_size=0.15, shuffle=False)

# 3. Dataset y Data Loader

In [12]:
# ============================================
# 3. Dataset & DataLoader
# ============================================
class BikeDataset(Dataset):
    def __init__(self, data):
        self.x_cat  = torch.tensor(data[cat_cols ].values, dtype=torch.long)
        self.x_cont = torch.tensor(data[cont_cols].values, dtype=torch.float32)
        self.y_cls  = torch.tensor(data[target_cls].values, dtype=torch.long)
        self.y_eta  = torch.tensor(data[target_eta].values, dtype=torch.float32)
    def __len__(self): return len(self.y_cls)
    def __getitem__(self, idx):
        return (self.x_cat[idx], self.x_cont[idx], self.y_cls[idx], self.y_eta[idx])

In [13]:
BATCH = 512
train_loader = DataLoader(BikeDataset(train_df), batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(BikeDataset(val_df),   batch_size=BATCH, shuffle=False, num_workers=0)
test_loader  = DataLoader(BikeDataset(test_df),  batch_size=BATCH, shuffle=False, num_workers=0)

# 4. Instanciar Modelo

In [14]:
# ============================================
# 4. Instanciar modelo
# ============================================
cat_dims = {c: int(df[c].max()) + 1 for c in cat_cols}
model = BikeDestETAEnhanced(cat_dims=cat_dims,
                            cont_dim=len(cont_cols),
                            num_stations=int(df[target_cls].max()) + 1).to(DEVICE)

## 4.1 Pesos de las clases

In [15]:
# ---------- 4.1 Pesos de clases (desbalance) ----------
hist = np.bincount(train_df[target_cls].values,
                   minlength=model.dest_head[-1].out_features)
class_w = torch.tensor(1 / np.sqrt(hist + 1e-6), dtype=torch.float32)
class_w = class_w / class_w.mean()   # normalizá si querés
class_w = class_w.to(DEVICE)

# 5. Scheduler y Adam

In [16]:
# ============================================
# 5. Optimizador y scheduler
# ============================================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

# 6. Loop de Entrenamiento

In [17]:
# ============================================
# 6. Loop de entrenamiento
# ============================================
EPOCHS = 30
best_val_f1, best_state = 0, None

for epoch in range(1, EPOCHS + 1):
    # ---------- 6.1 Entrenamiento ----------
    model.train()
    for x_cat, x_cont, y_cls, y_eta in train_loader:
        x_cat, x_cont = x_cat.to(DEVICE), x_cont.to(DEVICE)
        y_cls, y_eta  = y_cls.to(DEVICE), y_eta.to(DEVICE)

        dest_log, eta_mu, eta_sigma = model(x_cat, x_cont)
        loss, _, _ = model.loss(dest_log, y_cls, eta_mu, eta_sigma, y_eta,
                                class_weights=class_w)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    scheduler.step()

    # ---------- 6.2 Validación ----------
    model.eval()
    correct, total, mae, n = 0, 0, 0.0, 0
    with torch.no_grad():
        for x_cat, x_cont, y_cls, y_eta in val_loader:
            x_cat, x_cont = x_cat.to(DEVICE), x_cont.to(DEVICE)
            y_cls, y_eta  = y_cls.to(DEVICE), y_eta.to(DEVICE)

            dest_log, eta_mu, _ = model(x_cat, x_cont)
            pred_cls = dest_log.argmax(1)
            correct += (pred_cls == y_cls).sum().item()
            total   += y_cls.size(0)

            mae   += (eta_mu - y_eta).abs().sum().item()
            n     += y_eta.size(0)

    acc = correct / total
    mae = mae / n
    print(f"Epoch {epoch:02d} | Val Acc={acc*100:.2f}% | Val MAE={mae:.2f} min")

    # ---------- 6.3 Guardar mejor modelo ----------
    if acc > best_val_f1:             # usá F1 si lo calculás
        best_val_f1 = acc
        best_state  = model.state_dict()


Epoch 01 | Val Acc=0.08% | Val MAE=861.13 min
Epoch 02 | Val Acc=0.00% | Val MAE=853.58 min
Epoch 03 | Val Acc=0.00% | Val MAE=840.60 min
Epoch 04 | Val Acc=0.52% | Val MAE=828.07 min
Epoch 05 | Val Acc=0.03% | Val MAE=830.09 min
Epoch 06 | Val Acc=0.15% | Val MAE=837.44 min
Epoch 07 | Val Acc=0.19% | Val MAE=830.94 min
Epoch 08 | Val Acc=0.06% | Val MAE=833.10 min
Epoch 09 | Val Acc=0.19% | Val MAE=838.32 min
Epoch 10 | Val Acc=0.00% | Val MAE=843.11 min
Epoch 11 | Val Acc=0.00% | Val MAE=832.97 min
Epoch 12 | Val Acc=0.12% | Val MAE=832.74 min
Epoch 13 | Val Acc=0.01% | Val MAE=839.97 min
Epoch 14 | Val Acc=0.24% | Val MAE=835.95 min
Epoch 15 | Val Acc=0.31% | Val MAE=834.20 min
Epoch 16 | Val Acc=0.31% | Val MAE=834.58 min
Epoch 17 | Val Acc=0.41% | Val MAE=834.56 min
Epoch 18 | Val Acc=0.31% | Val MAE=836.80 min
Epoch 19 | Val Acc=0.00% | Val MAE=835.77 min
Epoch 20 | Val Acc=0.09% | Val MAE=832.46 min
Epoch 21 | Val Acc=0.02% | Val MAE=847.57 min
Epoch 22 | Val Acc=0.03% | Val MAE

# 7 y 8. Evaluar y Guardar

In [19]:
# ============================================
# 7. Evaluación final en test
# ============================================
model.load_state_dict(best_state)
model.eval()


# ============================================
# 8. Serializar modelo y encoders
# ============================================
torch.save({
    'state_dict': model.state_dict(),
    'cat_dims'  : cat_dims,
    'encoders'  : encoders           # usá joblib si preferís
}, "bike_multi_task.pth")

# 9. Comprehensive Metrics Evaluation


In [20]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, top_k_accuracy_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# 9. Comprehensive Metrics Evaluation
# ============================================

# Load best model
model.load_state_dict(best_state)
model.eval()

# Initialize containers for predictions and true values
all_pred_cls = []
all_true_cls = []
all_pred_eta = []
all_true_eta = []

print("Evaluating on test set...")
with torch.no_grad():
    for x_cat, x_cont, y_cls, y_eta in test_loader:
        x_cat, x_cont = x_cat.to(DEVICE), x_cont.to(DEVICE)
        y_cls, y_eta = y_cls.to(DEVICE), y_eta.to(DEVICE)
        
        dest_log, eta_mu, _ = model(x_cat, x_cont)
        pred_cls = dest_log.argmax(1)
        
        # Store predictions and true values
        all_pred_cls.extend(pred_cls.cpu().numpy())
        all_true_cls.extend(y_cls.cpu().numpy())
        all_pred_eta.extend(eta_mu.cpu().numpy())
        all_true_eta.extend(y_eta.cpu().numpy())

# Convert to numpy arrays
all_pred_cls = np.array(all_pred_cls)
all_true_cls = np.array(all_true_cls)
all_pred_eta = np.array(all_pred_eta)
all_true_eta = np.array(all_true_eta)

print("Test set evaluation complete!")


Evaluating on test set...
Test set evaluation complete!


## 9.1 Classification Metrics (Destination Prediction)


In [21]:
# ============================================
# 9.1 Classification Metrics (Destination Prediction)
# ============================================

print("=== DESTINATION PREDICTION METRICS ===")
print(f"Test Accuracy: {accuracy_score(all_true_cls, all_pred_cls):.4f}")

# Top-k accuracy for destination prediction
with torch.no_grad():
    all_probs = []
    for x_cat, x_cont, y_cls, y_eta in test_loader:
        x_cat, x_cont = x_cat.to(DEVICE), x_cont.to(DEVICE)
        dest_log, _, _ = model(x_cat, x_cont)
        probs = torch.softmax(dest_log, dim=1)
        all_probs.extend(probs.cpu().numpy())
    
    all_probs = np.array(all_probs)

# Calculate top-k accuracies
for k in [1, 3, 5, 10]:
    top_k_acc = top_k_accuracy_score(all_true_cls, all_probs, k=k)
    print(f"Top-{k} Accuracy: {top_k_acc:.4f}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(all_true_cls, all_pred_cls, zero_division=0))

# Most common prediction errors
print("\n=== MOST COMMON PREDICTION ERRORS ===")
errors = all_true_cls != all_pred_cls
error_pairs = list(zip(all_true_cls[errors], all_pred_cls[errors]))
from collections import Counter
most_common_errors = Counter(error_pairs).most_common(10)
print("True -> Predicted (Count)")
for (true_cls, pred_cls), count in most_common_errors:
    print(f"{true_cls} -> {pred_cls} ({count})")


=== DESTINATION PREDICTION METRICS ===
Test Accuracy: 0.0010


ValueError: Input contains NaN.

## 9.2 Regression Metrics (ETA Prediction)


In [ ]:
# ============================================
# 9.2 Regression Metrics (ETA Prediction)
# ============================================

print("=== ETA PREDICTION METRICS ===")

# Basic regression metrics
mae = mean_absolute_error(all_true_eta, all_pred_eta)
mse = mean_squared_error(all_true_eta, all_pred_eta)
rmse = np.sqrt(mse)
r2 = r2_score(all_true_eta, all_pred_eta)

print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} minutes")
print(f"R² Score: {r2:.4f}")

# Additional metrics
mape = np.mean(np.abs((all_true_eta - all_pred_eta) / all_true_eta)) * 100
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

# Percentile-based metrics
errors = np.abs(all_true_eta - all_pred_eta)
print(f"\n=== ERROR DISTRIBUTION ===")
print(f"50th percentile (median) error: {np.percentile(errors, 50):.2f} minutes")
print(f"75th percentile error: {np.percentile(errors, 75):.2f} minutes")
print(f"90th percentile error: {np.percentile(errors, 90):.2f} minutes")
print(f"95th percentile error: {np.percentile(errors, 95):.2f} minutes")

# Accuracy within time windows
within_5min = np.mean(errors <= 5) * 100
within_10min = np.mean(errors <= 10) * 100
within_15min = np.mean(errors <= 15) * 100
within_30min = np.mean(errors <= 30) * 100

print(f"\n=== PREDICTION ACCURACY WITHIN TIME WINDOWS ===")
print(f"Within 5 minutes: {within_5min:.1f}%")
print(f"Within 10 minutes: {within_10min:.1f}%")
print(f"Within 15 minutes: {within_15min:.1f}%")
print(f"Within 30 minutes: {within_30min:.1f}%")


## 9.3 Visualization


In [ ]:
# ============================================
# 9.3 Visualization
# ============================================

plt.figure(figsize=(15, 10))

# Plot 1: ETA Prediction Scatter Plot
plt.subplot(2, 3, 1)
sample_size = min(10000, len(all_true_eta))  # Sample for better visualization
idx = np.random.choice(len(all_true_eta), sample_size, replace=False)
plt.scatter(all_true_eta[idx], all_pred_eta[idx], alpha=0.5, s=1)
plt.plot([all_true_eta.min(), all_true_eta.max()], [all_true_eta.min(), all_true_eta.max()], 'r--', lw=2)
plt.xlabel('True ETA (minutes)')
plt.ylabel('Predicted ETA (minutes)')
plt.title('ETA Prediction: True vs Predicted')
plt.grid(True, alpha=0.3)

# Plot 2: Error Distribution
plt.subplot(2, 3, 2)
errors = all_true_eta - all_pred_eta
plt.hist(errors, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Prediction Error (minutes)')
plt.ylabel('Frequency')
plt.title('ETA Prediction Error Distribution')
plt.axvline(x=0, color='red', linestyle='--', alpha=0.7)
plt.grid(True, alpha=0.3)

# Plot 3: Absolute Error Distribution
plt.subplot(2, 3, 3)
abs_errors = np.abs(errors)
plt.hist(abs_errors, bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Absolute Error (minutes)')
plt.ylabel('Frequency')
plt.title('Absolute ETA Prediction Error')
plt.grid(True, alpha=0.3)

# Plot 4: Error vs True ETA
plt.subplot(2, 3, 4)
plt.scatter(all_true_eta[idx], abs_errors[idx], alpha=0.5, s=1)
plt.xlabel('True ETA (minutes)')
plt.ylabel('Absolute Error (minutes)')
plt.title('Absolute Error vs True ETA')
plt.grid(True, alpha=0.3)

# Plot 5: Top 20 most frequent true destinations
plt.subplot(2, 3, 5)
top_destinations = pd.Series(all_true_cls).value_counts().head(20)
plt.bar(range(len(top_destinations)), top_destinations.values)
plt.xlabel('Destination Station (Encoded)')
plt.ylabel('Frequency')
plt.title('Top 20 Most Frequent Destinations')
plt.xticks(range(len(top_destinations)), top_destinations.index, rotation=45)
plt.grid(True, alpha=0.3)

# Plot 6: Cumulative Error Distribution
plt.subplot(2, 3, 6)
sorted_abs_errors = np.sort(abs_errors)
cumulative_pct = np.arange(1, len(sorted_abs_errors) + 1) / len(sorted_abs_errors) * 100
plt.plot(sorted_abs_errors, cumulative_pct)
plt.xlabel('Absolute Error (minutes)')
plt.ylabel('Cumulative Percentage (%)')
plt.title('Cumulative Distribution of Absolute Errors')
plt.grid(True, alpha=0.3)
plt.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='50th percentile')
plt.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='90th percentile')
plt.legend()

plt.tight_layout()
plt.show()

# Summary Statistics
print(f"\n=== FINAL SUMMARY ===")
print(f"Dataset Size: {len(all_true_eta):,} test samples")
print(f"Unique Destinations: {len(np.unique(all_true_cls))}")
print(f"Destination Prediction Accuracy: {accuracy_score(all_true_cls, all_pred_cls):.4f}")
print(f"ETA MAE: {mae:.2f} minutes")
print(f"ETA RMSE: {rmse:.2f} minutes")
print(f"Predictions within 10 minutes: {within_10min:.1f}%")
